In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import re
from pathlib import Path
import torch
from tqdm import tqdm
import pickle
import json
from collections import defaultdict
from utils import model_loader

data_dir = Path("./data")
save_dir = Path("./results/5000_samples_ProverQA_Llama3.1-8B")

model_name = "meta-llama/Llama-3.1-8B-Instruct"
model, tokenizer = model_loader(model_name)

json_path = save_dir / "logical_result_standard.json"
with open(json_path, "r") as f:
    data = json.load(f)

/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.86it/s]


In [3]:
system_prompt = "Given a problem statement as contexts, the task is to answer a logical reasoning question. Your answer should be in JSON format with key: answer."
if 'reasoning' in json_path.stem:
    in_context_examples_for_train = "Context:\nAnyone who can solve problems and communicate effectively has practical skills. If Paola appreciates beauty, then she values precision and enjoys manual work. Paola appreciates beauty. Paola is either skilled in mathematics or has practical skills, but not both. If someone enjoys manual work and values attention to detail, then they can craft with their hands. Paola is either skilled in mathematics or artistically inclined, but surprisingly, she is actually both. Paola values attention to detail. Anyone who crafts with their hands has practical skills.\n\nQuestion: Based on the above information, is the following statement true, false, or uncertain? Paola is artistically inclined.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"reasoning\": \"Paola appreciates beauty. If Paola appreciates beauty, then she values precision and enjoys manual work. Paola enjoys manual work. Paola enjoys manual work. Paola values attention to detail. If someone enjoys manual work and values attention to detail, then they can craft with their hands. Paola crafts with her hands. Paola crafts with her hands. Anyone who crafts with their hands has practical skills. Paola has practical skills. Paola has practical skills. Paola is either skilled in mathematics or has practical skills, but not both. Paola is not skilled in mathematics. Paola is not skilled in mathematics. Paola is either skilled in mathematics or artistically inclined, but surprisingly, she is actually both. Paola is artistically inclined. Therefore, it is true that Paola is artistically inclined. The correct option is: A.\",\n  \"answer\": \"A\"\n}\n------\nContext:\nIf Priscilla loves nature or shares beautiful images (or both), then she inspires environmental awareness. Priscilla either portrays cityscapes or captures sunsets, but not both. Priscilla documents wildlife. If Priscilla has an urban focus, then she will portray cityscapes. If Priscilla travels extensively, then she either takes landscape photos or captures sunsets (or both). Priscilla has an urban focus. Everyone who either documents wildlife or captures sunsets (but not both) shares beautiful images.\n\nQuestion: Based on the above information, is the following statement true, false, or uncertain? Priscilla does not inspire environmental awareness.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"reasoning\": \"Priscilla has an urban focus. If Priscilla has an urban focus, then she will portray cityscapes. Priscilla portrays cityscapes. Priscilla portrays cityscapes. Priscilla either portrays cityscapes or captures sunsets, but not both. Priscilla does not capture sunsets. Priscilla documents wildlife. Priscilla does not capture sunsets. Everyone who either documents wildlife or captures sunsets (but not both) shares beautiful images. Priscilla shares beautiful images. Priscilla shares beautiful images. If Priscilla loves nature or shares beautiful images (or both), then she inspires environmental awareness. Priscilla inspires environmental awareness. Therefore, it is false that Priscilla does not inspire environmental awareness. The correct option is: B.\",\n  \"answer\": \"B\"\n}\n------\n[[INSTRUCTION]]\n\nThe correct option is:"
else:
    in_context_examples_for_train = "Context:\nAnyone who can solve problems and communicate effectively has practical skills. If Paola appreciates beauty, then she values precision and enjoys manual work. Paola appreciates beauty. Paola is either skilled in mathematics or has practical skills, but not both. If someone enjoys manual work and values attention to detail, then they can craft with their hands. Paola is either skilled in mathematics or artistically inclined, but surprisingly, she is actually both. Paola values attention to detail. Anyone who crafts with their hands has practical skills.\n\nQuestion: Based on the above information, is the following statement true, false, or uncertain? Paola is artistically inclined.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"answer\": \"A\"\n}\n------\nContext:\nIf Priscilla loves nature or shares beautiful images (or both), then she inspires environmental awareness. Priscilla either portrays cityscapes or captures sunsets, but not both. Priscilla documents wildlife. If Priscilla has an urban focus, then she will portray cityscapes. If Priscilla travels extensively, then she either takes landscape photos or captures sunsets (or both). Priscilla has an urban focus. Everyone who either documents wildlife or captures sunsets (but not both) shares beautiful images.\n\nBased on the above information, is the following statement true, false, or uncertain? Priscilla does not inspire environmental awareness.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"answer\": \"B\"\n}\n------\n[[INSTRUCTION]]\n\nThe correct option is:"
# in_context_examples_for_test = "Context:\nAnyone who can solve problems and communicate effectively has practical skills. If Paola appreciates beauty, then she values precision and enjoys manual work. Paola appreciates beauty. Paola is either skilled in mathematics or has practical skills, but not both. If someone enjoys manual work and values attention to detail, then they can craft with their hands. Paola is either skilled in mathematics or artistically inclined, but surprisingly, she is actually both. Paola values attention to detail. Anyone who crafts with their hands has practical skills.\n\nQuestion: Based on the above information, is the following statement true, false, or uncertain? Paola is artistically inclined.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"answer\": \"A\"\n}\n------\nContext:\nIf Priscilla loves nature or shares beautiful images (or both), then she inspires environmental awareness. Priscilla either portrays cityscapes or captures sunsets, but not both. Priscilla documents wildlife. If Priscilla has an urban focus, then she will portray cityscapes. If Priscilla travels extensively, then she either takes landscape photos or captures sunsets (or both). Priscilla has an urban focus. Everyone who either documents wildlife or captures sunsets (but not both) shares beautiful images.\n\nBased on the above information, is the following statement true, false, or uncertain? Priscilla does not inspire environmental awareness.\n\nOptions:\nA) True\nB) False\nC) Uncertain\n\nThe correct option is: {\n  \"answer\": \"B\"\n}\n------\nContext:\n[[CONTEXT]]\n\nQuestion: [[QUESTION]]\n\nOptions:\n[[OPTIONS]]\n\nThe correct option is:"

In [ ]:
results = defaultdict(list)
index = 0

for item in tqdm(data):
    if item["pred_label"]:
        instruction = item['instruction']
        full_prompt = in_context_examples_for_train.replace("[[INSTRUCTION]]", instruction)
        
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': full_prompt}
        ]
        trace = item["model_trace"]
        correct = item["correct_prediction"]
        results['correct'].append(correct)

        full_messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': full_prompt},
            {'role': 'assistant', 'content': trace}
        ]

        tokenized_query = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).to(model.device)
        prompt_length = tokenized_query.input_ids.shape[-1]

        full_inputs = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).input_ids.to(model.device)

        kwargs = {
            "input_ids": full_inputs,
            "use_cache": False,
            "past_key_values": None,
            "output_hidden_states": True,
            "return_dict": True,
        }
        with torch.no_grad():
            outputs = model(**kwargs)

        generated_embeddings = outputs.hidden_states[-1][0][prompt_length-1:-1].to(torch.float32).detach().cpu().numpy()
        generated_embeddings = np.stack(generated_embeddings)
        # print(f"Sample {index}: Generated embeddings shape: {generated_embeddings.shape}")
        results['generated_embeddings'].append(generated_embeddings)
        results['index'].append(index)
    index += 1
with open(save_dir / "contextual_embeddings_standard_every_gen_tokens.pkl", "wb") as outfile:
    outfile.write(pickle.dumps(results))

Classifier

In [1]:
from model import *
import os
import torch
import numpy as np
from utils import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

classifier = SingleMLP_Classifier
results_file = "./results/5000_samples_ProverQA_Llama3.1-8B/contextual_embs_standard_all_gen_tokens.pkl"

/data/share/project/RSML/llm-hallucinations-factual-qa/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:0


In [2]:
from sklearn.model_selection import train_test_split
from tqdm import tqdm

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2
save_model_dir = "./models/Llama-3.1-8B-Instruct"
save_name = "mlp_context_emb_ProverQA.pt"
os.makedirs(save_model_dir, exist_ok=True)

results, correct = load_hidden_states(results_file)
labels = correct
pooled_embeddings = []
for i in range(len(results['generated_embeddings'])):
    embeddings = results['generated_embeddings'][i]
    emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
    pooled_emb = torch.mean(emb_sequence, dim=0)    # mean pooling of the embeddings for each sample
    # pooled_emb = torch.amax(emb_sequence, dim=0)    # max pooling
    pooled_embeddings.append(pooled_emb)
inputs = torch.stack(pooled_embeddings)
indices = range(len(inputs))

X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(inputs, labels.astype(int), indices, test_size=0.2, random_state=123)
classifier_model = classifier(X_train.shape[1]).to(device)
X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(torch.long).to(device)
X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(torch.long).to(device)

optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

for _ in tqdm(range(1001)):
    optimizer.zero_grad()
    sample = torch.randperm(X_train.shape[0])[:batch_size]
    pred = classifier_model(X_train[sample])
    loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
    loss.backward()
    optimizer.step()
torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

classifier_model.eval()
with torch.no_grad():
    pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
    prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()

context_emb_roc, context_emb_acc = roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

Using device: cuda:0


/tmp/ipykernel_187943/1573214895.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_187943/1573214895.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:06<00:00, 163.80it/s]


Contextual Embeddings ROC AUC: 0.7763, Accuracy: 0.7077


In [ ]:
from pathlib import Path
save_dir = Path("./results/5000_samples_ProverQA_Llama3.1-8B")

with open(save_dir / "score_result_standard.txt", "a") as text_file:
    text_file.write(f"Results for {str(classifier)}:\n")
    text_file.write("Contextual Embeddings ROC AUC: " + str(context_emb_roc) + "\n")
    text_file.write("Contextual Embeddings Accuracy: " + str(context_emb_acc) + "\n")

Visualization

In [4]:
def compute_confusion_stats(y_true, y_pred, labels=(0, 1)):
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    tn, fp, fn, tp = cm.ravel()
    stats = {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "accuracy": float((tp + tn) / cm.sum()),
        "precision": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
        "recall": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
    }
    return stats, cm


def plot_confusion_matrix(cm, labels=("Hallucinated", "Non-hallucinated"), save_path="./results/confusion_matrix.png", label=None):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    ax.set_facecolor("white")

    cmap = plt.cm.Blues
    img = ax.imshow(cm, cmap=cmap, vmin=0, vmax=max(1, cm.max()))

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_yticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_xlabel("Predicted label", color="#0f172a", fontsize=11)
    ax.set_ylabel("True label", color="#0f172a", fontsize=11)
    ax.set_title(f"{label} Confusion Matrix", color="#0f172a", fontsize=12, pad=10)
    ax.tick_params(axis="both", colors="#0f172a")
    for spine in ax.spines.values():
        spine.set_color("#1f2937")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            cell_color = cmap(img.norm(cm[i, j]))
            brightness = 0.299 * cell_color[0] + 0.587 * cell_color[1] + 0.114 * cell_color[2]
            text_color = "white" if brightness < 0.55 else "#111827"
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color=text_color, fontsize=11, fontweight="bold")

    cbar = fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_tick_params(color="#0f172a")
    plt.setp(cbar.ax.get_yticklabels(), color="#0f172a")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    # plt.close(fig)
    return save_path

In [5]:
pred_labels = prediction_classes.numpy()
true_labels = y_test.cpu().numpy()
cm = confusion_matrix(true_labels, pred_labels)
confusion_plot_path = plot_confusion_matrix(
    cm,
    labels=("Hallucinated", "Non-hallucinated"),
    save_path="./results/5000_samples_ProverQA_Llama3.1-8B/confusion_matrix_hallu.png",
    label="Hallucinations Classifier"
)